# 02 — Train and evaluate YOLOv8s on facade materials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarEAbdelaal/M4U3-CV-Facade-Materials/blob/main/notebooks/02_training_eval.ipynb)

Detects 6 facade materials: `brick`, `cladding_panel`, `exposed_concrete`, `glass`, `painted_render`, `stone_cladding`.

**Reproducible with no credentials.** The cleaned dataset (Release `v1.1`) is downloaded from the project's GitHub Release and checked against its SHA256. No Roboflow account, API key or Colab Secret is needed.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

| Cell | What it does |
|---|---|
| 1 | Install the pinned Ultralytics version |
| 2 | Settings (model, epochs, image size, new-image URLs) |
| 3 | Download the frozen dataset from the GitHub Release and verify SHA256 |
| 3b | Optional: Roboflow download (skipped automatically when cell 3 worked) |
| 4 | Check the labels and count boxes per class and split |
| 5 | Record the environment (versions, GPU, time) |
| 6 | Train YOLOv8s |
| 7 | Evaluate: precision / recall / mAP50 / mAP50-95, overall and per class |
| 8 | Save training curves and confusion matrix |
| 9 | Predict on 10 validation images |
| 10 | Predict on new images that are not in the dataset |
| 11 | Package results and weights for download |
| 12 | Compare with Run 03 (uncleaned v1.0) and any other runs in this session |

Outputs go to `/content/results/<RUN_NAME>/`, matching the repo's `results/` folder.

**Dataset.** Release `v1.1` is the Roboflow version-4 export after the label cleaning in `01b_clean_dataset.ipynb`: polygons converted to boxes, fragments removed, one box per continuous area, stacked labels resolved, boxes under 1% of the image removed, and 24 images without a label of their own material removed. Run 03 (first full run) used the uncleaned Release `v1.0`.

In [ ]:
# Cell 1 — install (version pinned for reproducibility)
%pip install -q ultralytics==8.4.163

In [ ]:
# Cell 2 — settings
from pathlib import Path

MODEL = "yolov8s.pt"      # pretrained on COCO; fine-tuned here (transfer learning)
EPOCHS = 50
IMGSZ = 640
BATCH = 16
SEED = 0
FRACTION = 1.0            # share of training images to use (1.0 = all)

# If no GPU is available: set QUICK_RUN = True for a 5-epoch verification run,
# and put the Release URL of the real best.pt in WEIGHTS_URL so evaluation and
# inference use the fully trained model (allowed by the brief; say so in the README).
QUICK_RUN = False
WEIGHTS_URL = ""   # published weights: https://github.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/releases/download/weights-v1/best.pt
if QUICK_RUN:
    EPOCHS = 5

CONF = 0.25               # confidence threshold for the prediction images

# 5 images that are NOT in the dataset (e.g. Wikimedia Commons file URLs), for the new-image evidence
BASE_NEW = "https://raw.githubusercontent.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/main/data/new_images/"
# Copies are kept in data/new_images/ (GitHub, no rate limit); original Wikimedia pages and licenses in data/new_images.csv
NEW_IMAGE_URLS = [
    BASE_NEW + "new_00_brick.jpg",
    BASE_NEW + "new_01_glass.jpg",
    BASE_NEW + "new_02_exposed_concrete.jpg",
    BASE_NEW + "new_03_cladding_panel.jpg",
    BASE_NEW + "new_04_stone_cladding.jpg",
]

RUN_NAME = "yolov8s_clean"   # results go to /content/results/<RUN_NAME>/

RESULTS = Path("/content/results") / RUN_NAME
for sub in ["curves", "evidence/val_predictions", "evidence/new_predictions", "weights"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)
print(f"Run: {RUN_NAME} | {MODEL}, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, quick_run={QUICK_RUN}")

## Dataset (primary, keyless path)

This is the course's standard download cell. Only `DATA_URL` and `SHA256` are project-specific.

- Release: `v1.1`, asset `facade-materials-v4-clean-yolov8.zip`: Roboflow project `facade-materials-3mrrt` **version 4**, cleaned by `01b_clean_dataset.ipynb` from Release `v1.0`
- The checksum proves the file is the exact one the model was trained on.

In [ ]:
# Cell 3 — download the frozen dataset from the GitHub Release and verify it
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/releases/download/v1.1/facade-materials-v4-clean-yolov8.zip"
SHA256 = "bdcf62ff01fd58892ee22dd07cf2b481fa1194d0946dd5f78c6b41dc04187164"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

## Dataset (optional secondary path: Roboflow)

Kept for the author's own workflow. On **Run all** it does nothing when the keyless dataset from cell 3 is already in place. Roboflow holds the *uncleaned* version 4; data downloaded this way must go through `01b_clean_dataset.ipynb` before training. It never crashes when no API key is set (it asks for one instead), and it is pinned to **version 4**, never "latest".

Roboflow's `yolov8` and `yolov11` exports use the same label and `data.yaml` format; `yolov8` matches the brief.

In [ ]:
# Cell 3b — secondary download from Roboflow (skipped when cell 3 succeeded)
from pathlib import Path

if not list(Path("/content/dataset").rglob("data.yaml")):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
    try:
        from google.colab import userdata
        api_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        from getpass import getpass
        api_key = getpass("Roboflow API key (any account works for public Universe datasets): ")

    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    rf.workspace("omar-el-sayed-pzb69").project("facade-materials-3mrrt").version(4).download(
        "yolov8", location="/content/dataset_roboflow"
    )
    print("Downloaded to /content/dataset_roboflow. Point cfg_path at its data.yaml to use it.")
else:
    print("Keyless dataset already present; skipping Roboflow download.")

## Label check

The labels in Release `v1.1` are already clean. This cell only checks that every line is a valid box, and counts boxes and images per class and split.

In [ ]:
# Cell 4 — check the labels and count them
import collections
import pandas as pd

names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
WORK, work_cfg_path = dataset_root, cfg_path      # the cleaned release is used as it is

bad, rows = 0, []
for split in ["train", "valid", "test"]:
    lbl_dir = dataset_root / split / "labels"
    if not lbl_dir.is_dir():
        continue
    boxes, imgs = collections.Counter(), collections.Counter()
    files = sorted(lbl_dir.glob("*.txt"))
    for lbl in files:
        seen = set()
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) != 5 or not all(0.0 <= float(v) <= 1.0 for v in p[1:]):
                bad += 1
                continue
            boxes[names[int(p[0])]] += 1
            seen.add(names[int(p[0])])
        for c in seen:
            imgs[c] += 1
    for c in names:
        rows.append({"split": split, "class": c, "images": imgs[c], "boxes": boxes[c]})
    print(f"{split}: {len(files)} images")
assert bad == 0, f"{bad} invalid label lines: is this the cleaned release?"

counts = pd.DataFrame(rows).pivot(index="class", columns="split", values=["images", "boxes"])
counts.to_csv(RESULTS / "dataset_counts.csv")
counts

In [ ]:
# Cell 5 — record the environment (for the README "Reproducibility proof")
import json, platform, datetime, torch, ultralytics

run_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
    "quick_run": QUICK_RUN, "dataset_url": DATA_URL, "dataset_sha256": SHA256,
    "run_name": RUN_NAME, "dataset_release": "v1.1 (cleaned)",
}
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(json.dumps(run_info, indent=2))
if not torch.cuda.is_available():
    print("\nNo GPU found. Runtime -> Change runtime type -> T4 GPU, or set QUICK_RUN = True in cell 2.")

In [ ]:
# Cell 6 — train
import time
from ultralytics import YOLO

t0 = time.time()
model = YOLO(MODEL)
model.train(
    data=str(work_cfg_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
    deterministic=True, fraction=FRACTION, plots=True,
    project="/content/runs", name=RUN_NAME, exist_ok=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
train_minutes = (time.time() - t0) / 60
print(f"Training took {train_minutes:.1f} min. Best weights: {BEST}")

if QUICK_RUN and WEIGHTS_URL:
    BEST = Path("/content/best_published.pt")
    urllib.request.urlretrieve(WEIGHTS_URL, BEST)
    print("QUICK_RUN: evaluation and inference use the published weights:", WEIGHTS_URL)

run_info["train_minutes"] = round(train_minutes, 1)
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))

In [ ]:
# Cell 7 — evaluate: overall and per-class precision / recall / mAP
best = YOLO(str(BEST))
rows = []
for split in ["val"] + (["test"] if "test" in cfg else []):
    m = best.val(data=str(work_cfg_path), split=split, imgsz=IMGSZ, batch=BATCH, plots=True,
                 project="/content/runs", name=f"eval_{RUN_NAME}_{split}", exist_ok=True, verbose=False)
    rows.append({"run": RUN_NAME, "split": split, "class": "all", "precision": m.box.mp, "recall": m.box.mr,
                 "mAP50": m.box.map50, "mAP50-95": m.box.map})
    for i, c in enumerate(m.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({"run": RUN_NAME, "split": split, "class": names[int(c)], "precision": p, "recall": r,
                     "mAP50": ap50, "mAP50-95": ap})
metrics = pd.DataFrame(rows).round(3)
metrics.to_csv(RESULTS / "metrics.csv", index=False)
metrics

In [ ]:
# Cell 8 — save training curves and confusion matrices
import shutil
from IPython.display import Image as IPImage, display

for f in list(RUN_DIR.glob("*.png")) + list(RUN_DIR.glob("*.csv")):
    shutil.copy(f, RESULTS / "curves" / f.name)
for split in ["val", "test"]:
    ev = Path("/content/runs") / f"eval_{RUN_NAME}_{split}"
    for f in ev.glob("confusion_matrix*.png"):
        shutil.copy(f, RESULTS / "curves" / f"{split}_{f.name}")
print(sorted(p.name for p in (RESULTS / "curves").iterdir()))

for name in ["results.png", "confusion_matrix_normalized.png"]:
    if (RESULTS / "curves" / name).exists():
        display(IPImage(filename=str(RESULTS / "curves" / name), width=900))

In [ ]:
# Cell 9 — predictions on 10 validation images (same 10 every run)
import random
from PIL import Image

val_images = sorted((WORK / "valid" / "images").glob("*"))
random.Random(SEED).shuffle(val_images)
for img in val_images[:10]:
    r = best.predict(str(img), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    Image.fromarray(r.plot()[:, :, ::-1]).save(RESULTS / "evidence" / "val_predictions" / f"{img.stem}_pred.jpg")
print(len(list((RESULTS / "evidence" / "val_predictions").glob("*.jpg"))), "validation predictions saved")

In [ ]:
# Cell 10 — predictions on new images that are not in the dataset
import requests, time

new_dir = Path("/content/new_images")
new_dir.mkdir(exist_ok=True)
headers = {"User-Agent": "M4U3-facade-materials/1.0 (student project)"}
for i, url in enumerate(NEW_IMAGE_URLS):
    dst = new_dir / Path(url).name
    for attempt in range(5):
        if dst.exists():
            break
        try:
            resp = requests.get(url, headers=headers, timeout=60)
            if resp.status_code == 429:             # rate limited: wait and retry
                time.sleep(5 * (attempt + 1))
                continue
            resp.raise_for_status()
            dst.write_bytes(resp.content)
        except requests.RequestException as e:
            print(f"Could not download {url}: {e}")
            break
    time.sleep(1)

new_images = sorted(new_dir.glob("*.jpg"))
if not new_images:
    print("No new images: add 5 URLs to NEW_IMAGE_URLS in cell 2 (or upload files to /content/new_images).")
for img in new_images:
    r = best.predict(str(img), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    Image.fromarray(r.plot()[:, :, ::-1]).save(RESULTS / "evidence" / "new_predictions" / f"{img.stem}_pred.jpg")
print(len(list((RESULTS / "evidence" / "new_predictions").glob("*.jpg"))), "new-image predictions saved")

In [ ]:
# Cell 11 — package results and weights
shutil.copy(BEST, RESULTS / "weights" / "best.pt")
package = shutil.make_archive(f"/content/results_{RUN_NAME}", "zip", RESULTS)
print("results ->", package)
print("weights ->", RESULTS / "weights" / "best.pt", " (attach to a GitHub Release; do not commit it)")
try:
    from google.colab import files
    files.download(package)
except Exception:
    pass

In [ ]:
# Cell 12 — compare with earlier runs
# Run 03 used the uncleaned Release v1.0 and was scored on its uncleaned labels, so it is a
# reference point rather than a like-for-like comparison.
reference = pd.DataFrame([
    {"run": "03 v1.0 uncleaned", "split": "val",  "precision": 0.372, "recall": 0.159, "mAP50": 0.143, "mAP50-95": 0.093},
    {"run": "03 v1.0 uncleaned", "split": "test", "precision": 0.332, "recall": 0.216, "mAP50": 0.219, "mAP50-95": 0.141},
])
runs = [pd.read_csv(f) for f in sorted(Path("/content/results").glob("*/metrics.csv"))]
current = pd.concat(runs)
current = current[current["class"] == "all"][["run", "split", "precision", "recall", "mAP50", "mAP50-95"]]
comparison = pd.concat([reference, current]).set_index(["split", "run"]).sort_index()
comparison.to_csv("/content/results/comparison.csv")
comparison